# Example 2: Classification task on Glass Identification Dataset using a rule_reduced_ANFIS model with a custom training strategy

In [1]:
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report
)

import torch
import torch.nn as nn
import torch.utils.data as data

import numpy as np
import random

import neuro_fuzzy_toolbox as nft

In [2]:
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Data

In [3]:
glass_identification = fetch_ucirepo(id=42)

X = glass_identification.data.features 
y = glass_identification.data.targets

In [4]:
y['Type_of_glass'].unique().tolist()

[1, 2, 3, 5, 6, 7]

In [5]:
le = LabelEncoder()
y.loc[:, 'Type_of_glass'] = le.fit_transform(y['Type_of_glass'])
y = y.astype('int64')

In [6]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)

x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.2, stratify=y_train, random_state=SEED
)

In [7]:
scaler = MinMaxScaler(feature_range=(0, 1))

x_train = torch.from_numpy(scaler.fit_transform(x_train)).to(torch.float32)
x_val = torch.from_numpy(scaler.transform(x_val)).to(torch.float32)
x_test = torch.from_numpy(scaler.transform(x_test)).to(torch.float32)

y_train = torch.from_numpy(y_train.values).squeeze()
y_val = torch.from_numpy(y_val.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

## DataLoaders

In [8]:
generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size=8, shuffle=True, generator=generator)

val_loader = data.DataLoader(data.TensorDataset(x_val, y_val), batch_size=8, shuffle=False)

## Model

In [9]:
features = X.columns.tolist()
features

['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe']

In [10]:
# Define model
model = nft.rule_reduced_ANFIS(
    input_size = x_train.shape[1],
    num_mfs = 5, # 5 rules initially
    outputs = 6,
    membership_function=nft.GeneralizedBell_MF(),
    output_type='softmax',
    features=features,
)

In [11]:
#model.init_consequents(x_train, y_train, ridge_lambda=1.0)

## Learning Algorithm

In [12]:
# Define training strategy
trainer = nft.Basic_optimizer_training_algorithm(
    epochs=5000,
    loss_function=nn.CrossEntropyLoss(),
    optimizer=torch.optim.AdamW,
    optimizer_params={'lr': 1e-3, 'weight_decay': 1e-2},
    early_stopping=nft.EarlyStopping(patience=60)
)

In [13]:
# Train model
trainer(model, train_loader, val_loader)

/home/jsuarez/workspaces/neuro-fuzzy-toolbox/env/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch:    1/5000 - loss: 2.370941 - validation loss: 2.342811
Epoch:    2/5000 - loss: 2.191350 - validation loss: 2.139273
Epoch:    3/5000 - loss: 2.044862 - validation loss: 1.964481
Epoch:    4/5000 - loss: 1.962681 - validation loss: 1.867821
Epoch:    5/5000 - loss: 1.911191 - validation loss: 1.832808
Epoch:    6/5000 - loss: 1.868308 - validation loss: 1.797729
Epoch:    7/5000 - loss: 1.827927 - validation loss: 1.766327
Epoch:    8/5000 - loss: 1.790640 - validation loss: 1.734484
Epoch:    9/5000 - loss: 1.756359 - validation loss: 1.697627
Epoch:   10/5000 - loss: 1.723159 - validation loss: 1.665642
Epoch:   11/5000 - loss: 1.690833 - validation loss: 1.633132
Epoch:   12/5000 - loss: 1.659141 - validation loss: 1.597699
Epoch:   13/5000 - loss: 1.624445 - validation loss: 1.563603
Epoch:   14/5000 - loss: 1.586988 - validation loss: 1.520656
Epoch:   15/5000 - loss: 1.551092 - validation loss: 1.473819
Epoch:   16/5000 - loss: 1.517146 - validation loss: 1.427206
Epoch:  

## Initial evaluation

In [14]:
pred = model.predict(x_test)

acc = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred, average='weighted', zero_division=0)
recall = recall_score(y_test, pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_test, pred)
class_rep = classification_report(y_test, pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

print("Classification Report:")
print(class_rep)

Accuracy: 0.5846153846153846
Precision: 0.6485067873303167
Recall: 0.5846153846153846
f1 score: 0.5797903356799868 

Confusion Matrix:
[[ 8  6  6  0  1  0]
 [ 2 19  1  1  0  0]
 [ 0  4  1  0  0  0]
 [ 0  3  0  1  0  0]
 [ 0  0  0  0  2  1]
 [ 0  2  0  0  0  7]] 

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.38      0.52        21
           1       0.56      0.83      0.67        23
           2       0.12      0.20      0.15         5
           3       0.50      0.25      0.33         4
           4       0.67      0.67      0.67         3
           5       0.88      0.78      0.82         9

    accuracy                           0.58        65
   macro avg       0.59      0.52      0.53        65
weighted avg       0.65      0.58      0.58        65



## Custom Strategie: A simple greedy algorithm

In [15]:
"""
loss_fn = nn.CrossEntropyLoss()

def val_loss(model):
    with torch.no_grad():
        return sum(loss_fn(model(xb), yb)
                   for xb, yb in val_loader) / len(val_loader)

def train_with_patience(optimizer, epochs, patience):
    best, counter = val_loss(model), 0
    for _ in range(epochs):
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss_fn(model(xb), yb).backward()
            optimizer.step()
        current = val_loss(model)
        counter = 0 if current < best else counter + 1
        best = min(best, current)
        if counter >= patience: break

max_failed_attempts = 5
failed_attempts, best_loss = 0, val_loss(model)

while failed_attempts < max_failed_attempts:
    # Add a rule targeting the worst-recall class
    with torch.no_grad():
        recalls = recall_score(y_train, model.predict(x_train),
                               average=None, zero_division=0)
    worst = int(recalls.argmin())
    idx = (y_train == worst).nonzero(as_tuple=True)[0]
    idx = idx[torch.randint(0, len(idx), (1,))]
    model.add_rules(x_train[idx], torch.full_like(x_train[idx], 0.25))

    # Fine-tune only the new rule's parameters
    new_params = [model.get_premises_as_parameters_list()[-1],
                  model.get_consequents_as_parameters_list()[-1]]
    train_with_patience(
        torch.optim.AdamW(new_params, lr=0.005), 500, patience=30)

    if val_loss(model) < best_loss:
        # Global readaptation of all parameters
        train_with_patience(
            torch.optim.AdamW(model.parameters(), lr=0.001), 1000, patience=60)
        best_loss, failed_attempts = val_loss(model), 0
    else:
        model.remove_rules([model.rules - 1])
        failed_attempts += 1

print(f"Final number of rules: {model.rules}")
"""

'\nloss_fn = nn.CrossEntropyLoss()\n\ndef val_loss(model):\n    with torch.no_grad():\n        return sum(loss_fn(model(xb), yb)\n                   for xb, yb in val_loader) / len(val_loader)\n\ndef train_with_patience(optimizer, epochs, patience):\n    best, counter = val_loss(model), 0\n    for _ in range(epochs):\n        for xb, yb in train_loader:\n            optimizer.zero_grad()\n            loss_fn(model(xb), yb).backward()\n            optimizer.step()\n        current = val_loss(model)\n        counter = 0 if current < best else counter + 1\n        best = min(best, current)\n        if counter >= patience: break\n\nmax_failed_attempts = 5\nfailed_attempts, best_loss = 0, val_loss(model)\n\nwhile failed_attempts < max_failed_attempts:\n    # Add a rule targeting the worst-recall class\n    with torch.no_grad():\n        recalls = recall_score(y_train, model.predict(x_train),\n                               average=None, zero_division=0)\n    worst = int(recalls.argmin())\n 

In [16]:
loss_function = nn.CrossEntropyLoss()

In [17]:
def val_loss(model):
    with torch.no_grad():
        return sum(loss_function(model(xb), yb)
                   for xb, yb in val_loader) / len(val_loader)

In [18]:
# Hyperparameters for the greedy algorithm
max_failed_attempts = 5

single_adaptation_lr = 0.005
single_adaptation_epochs = 500
single_patience = 30

global_adaptation_lr = 0.001
global_adaptation_epochs = 1000
global_patience= 60

In [19]:
failed_attempts = 0
best_loss = val_loss(model)
print(f"Initial val loss: {best_loss:.4f} | Rules: {model.rules}")
print("=" * 60)

while failed_attempts < max_failed_attempts:
    # Identify worst-recall class
    with torch.no_grad():
        pred_train = model.predict(x_train)
    recalls = recall_score(y_train.numpy(), pred_train.numpy(),
                           average=None, zero_division=0)
    worst_class = int(recalls.argmin())
    print(f"Recalls per class: {[f'{r:.2f}' for r in recalls]}")
    print(f"Worst class: {worst_class} (recall={recalls[worst_class]:.2f})")

    # Add rule centered on a sample from the worst class
    class_indices = (y_train == worst_class).nonzero(as_tuple=True)[0]
    idx = class_indices[torch.randint(0, len(class_indices), (1,))]
    means = x_train[idx].to(torch.float32)
    stds  = torch.full_like(means, 0.25)
    model.add_rules(means, stds)
    print(f"Rule added. Total rules: {model.rules}")

    # Single-rule fine-tuning with early stopping
    new_params = [model.get_premises_as_parameters_list()[-1],
                  model.get_consequents_as_parameters_list()[-1]]
    opt_new = torch.optim.AdamW(new_params, lr=single_adaptation_lr,
                                weight_decay=0.01)
    best_single_loss = val_loss(model)
    patience_counter = 0

    for epoch in range(single_adaptation_epochs):
        for xb, yb in train_loader:
            opt_new.zero_grad()
            loss_function(model(xb), yb).backward()
            opt_new.step()
        current = val_loss(model)
        if current < best_single_loss:
            best_single_loss = current
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= single_patience:
            print(f"  Single adaptation stopped at epoch {epoch+1} "
                  f"| val loss: {current:.4f}")
            break

    val_after_single = val_loss(model)
    print(f"Val loss after single adaptation: {val_after_single:.4f} "
          f"(before: {best_loss:.4f})")

    # Retain or discard
    if val_after_single < best_loss:
        print("Rule RETAINED. Running global readaptation...")
        opt_all = torch.optim.AdamW(model.parameters(),
                                    lr=global_adaptation_lr, weight_decay=0.01)
        best_global_loss = val_after_single
        patience_counter = 0

        for epoch in range(global_adaptation_epochs):
            for xb, yb in train_loader:
                opt_all.zero_grad()
                loss_function(model(xb), yb).backward()
                opt_all.step()
            current = val_loss(model)
            if current < best_global_loss:
                best_global_loss = current
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= global_patience:
                print(f"  Global adaptation stopped at epoch {epoch+1} "
                      f"| val loss: {current:.4f}")
                break

        best_loss = val_loss(model)
        failed_attempts = 0
        print(f"Val loss after global adaptation: {best_loss:.4f}")
    else:
        model.remove_rules([model.rules - 1])
        failed_attempts += 1
        print(f"Rule DISCARDED. Failed attempts: {failed_attempts}/{max_failed_attempts}")

    print(f"Rules: {model.rules} | Best val loss: {best_loss:.4f}")
    print("-" * 60)

print(f"\nFinal number of rules: {model.rules}")

Initial val loss: 0.8744 | Rules: 5
Recalls per class: ['0.69', '0.79', '0.40', '1.00', '0.80', '0.81']
Worst class: 2 (recall=0.40)
Rule added. Total rules: 6


  Single adaptation stopped at epoch 33 | val loss: 0.8744
Val loss after single adaptation: 0.8744 (before: 0.8744)
Rule RETAINED. Running global readaptation...
  Global adaptation stopped at epoch 62 | val loss: 0.9219
Val loss after global adaptation: 0.9219
Rules: 6 | Best val loss: 0.9219
------------------------------------------------------------
Recalls per class: ['0.74', '0.71', '0.70', '1.00', '1.00', '0.81']
Worst class: 2 (recall=0.70)
Rule added. Total rules: 7
  Single adaptation stopped at epoch 34 | val loss: 0.9219
Val loss after single adaptation: 0.9219 (before: 0.9219)
Rule DISCARDED. Failed attempts: 1/5
Rules: 6 | Best val loss: 0.9219
------------------------------------------------------------
Recalls per class: ['0.74', '0.71', '0.70', '1.00', '1.00', '0.81']
Worst class: 2 (recall=0.70)
Rule added. Total rules: 7
  Single adaptation stopped at epoch 148 | val loss: 0.8924
Val loss after single adaptation: 0.8924 (before: 0.9219)
Rule RETAINED. Running global

## Final evaluation

In [20]:
pred = model.predict(x_test)

acc = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred, average='weighted', zero_division=0)
recall = recall_score(y_test, pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, pred, average='weighted', zero_division=0)
conf_matrix = confusion_matrix(y_test, pred)
class_rep = classification_report(y_test, pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

print("Classification Report:")
print(class_rep)

Accuracy: 0.6153846153846154
Precision: 0.6494505494505495
Recall: 0.6153846153846154
f1 score: 0.622604365590791 

Confusion Matrix:
[[12  3  5  0  1  0]
 [ 3 17  2  1  0  0]
 [ 1  3  1  0  0  0]
 [ 0  3  0  1  0  0]
 [ 0  0  0  0  2  1]
 [ 0  2  0  0  0  7]] 

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.57      0.65        21
           1       0.61      0.74      0.67        23
           2       0.12      0.20      0.15         5
           3       0.50      0.25      0.33         4
           4       0.67      0.67      0.67         3
           5       0.88      0.78      0.82         9

    accuracy                           0.62        65
   macro avg       0.59      0.53      0.55        65
weighted avg       0.65      0.62      0.62        65

